# Demand Forecaster — XGBoost (Booster API)

**Purpose**: Train a global demand forecasting model on the Rossmann Store Sales dataset.

**Dataset**: `data/rossmann/train.csv` + `store.csv`

**Target**: `units_sold` — 1-day ahead demand per store

**Pass criterion**: MAPE <= 15% on 20% hold-out test set

**Output**: `demand_global` saved to registry via `ModelStore`

**XGBoost 3.x**: uses `nthread` (not `n_jobs`), `tree_method='hist'` (not `gpu_hist`)

**Dev note**: `SAMPLE_FRAC=0.10` — Rossmann train.csv is 1M rows. Set to 1.0 for final run.

## 1 — Imports & Config

In [21]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error

from ml.features.demand_features import build_demand_features, fill_missing_dates, FEATURE_COLS
from ml.registry.model_store import ModelStore
from ml.shared.config_loader import _defaults

cfg            = _defaults()
FESTIVAL_DATES = cfg['festival_dates']
MAPE_TARGET    = cfg['mape_target']        # 0.15
XGB_CFG        = cfg['xgboost']

# Booster API params — nthread not n_jobs; no n_estimators/random_state keys
CV_PARAMS = {
    'objective':        'reg:squarederror',
    'max_depth':        XGB_CFG.get('max_depth',        6),
    'learning_rate':    XGB_CFG.get('learning_rate',    0.05),
    'subsample':        XGB_CFG.get('subsample',        0.8),
    'colsample_bytree': XGB_CFG.get('colsample_bytree', 0.8),
    'min_child_weight': XGB_CFG.get('min_child_weight', 3),
    'tree_method':      'hist',   # XGBoost 3.x: gpu_hist removed
    'nthread':          4,        # XGBoost 3.x: use nthread, not n_jobs
    'seed':             42,
    'verbosity':        0,
}
NUM_BOOST_ROUND = XGB_CFG.get('n_estimators', 500)

DATA_DIR = str((REPO_ROOT / 'data').resolve())
SAMPLE_FRAC = 1.0   # keep full time-series continuity for training

print(f'XGBoost version : {xgb.__version__}')
print(f'MAPE target     : {MAPE_TARGET}')
print(f'Sample fraction : {SAMPLE_FRAC}')
print(f'Data dir        : {DATA_DIR}')

XGBoost version : 3.2.0
MAPE target     : 0.15
Sample fraction : 1.0
Data dir        : C:\Users\Arghyadeep\Desktop\FlowSync\data


## 2 — Load Rossmann Dataset

In [22]:
train_path = os.path.join(DATA_DIR, 'rossmann', 'train.csv')
store_path = os.path.join(DATA_DIR, 'rossmann', 'store.csv')

for p in [train_path, store_path]:
    print(f"[{'OK' if os.path.exists(p) else 'MISSING'}] {p}")

assert os.path.exists(train_path), f'train.csv not found at {train_path}'
assert os.path.exists(store_path), f'store.csv not found at {store_path}'

train_raw = pd.read_csv(train_path, low_memory=False)
store_raw = pd.read_csv(store_path, low_memory=False)

if SAMPLE_FRAC < 1.0:
    sampled_stores = (
        train_raw['Store']
        .drop_duplicates()
        .sample(frac=SAMPLE_FRAC, random_state=42)
    )
    train_raw = train_raw[train_raw['Store'].isin(sampled_stores)].reset_index(drop=True)

df = train_raw.merge(store_raw, on='Store', how='left')
print(f'Loaded {len(df):,} rows ({SAMPLE_FRAC:.0%} store-level sample)')
print(df.dtypes)

[OK] C:\Users\Arghyadeep\Desktop\FlowSync\data\rossmann\train.csv
[OK] C:\Users\Arghyadeep\Desktop\FlowSync\data\rossmann\store.csv
Loaded 1,017,209 rows (100% store-level sample)
Store                          int64
DayOfWeek                      int64
Date                             str
Sales                          int64
Customers                      int64
Open                           int64
Promo                          int64
StateHoliday                     str
SchoolHoliday                  int64
StoreType                        str
Assortment                       str
CompetitionDistance          float64
CompetitionOpenSinceMonth    float64
CompetitionOpenSinceYear     float64
Promo2                         int64
Promo2SinceWeek              float64
Promo2SinceYear              float64
PromoInterval                    str
dtype: object


## 3 — Map to FlowSync Schema

Required by `build_demand_features`: `product_id, depot_id, date, units_sold, product_category, depot_region, is_cold_chain`

In [23]:
df = df.rename(columns={
    'Date':       'date',
    'Sales':      'units_sold',
    'StoreType':  'product_category',
    'Assortment': 'depot_region',
})
df['date']        = pd.to_datetime(df['date'], errors='coerce')
df['product_id']  = df['Store'].astype(str)
df['depot_id']    = df['Store'].astype(str)
df['is_cold_chain'] = False

df = df[(df['Open'] == 1) & (df['units_sold'] > 0)].copy()
df = df.sort_values(['depot_id', 'date']).reset_index(drop=True)
print(f'Rows after filtering: {len(df):,}')
df[['date','depot_id','product_id','units_sold','product_category','depot_region','is_cold_chain']].head(3)

Rows after filtering: 844,338


,date,depot_id,product_id,units_sold,product_category,depot_region,is_cold_chain
0,2013-01-02,1,1,5530,c,a,False
1,2013-01-03,1,1,4327,c,a,False
2,2013-01-04,1,1,4486,c,a,False


## 4 — Fill Missing Dates & Build Features

In [24]:
print('Filling missing dates...')
df_filled = fill_missing_dates(df)
print(f'Before: {len(df):,}  After: {len(df_filled):,}')

print('Building feature matrix...')
X, y = build_demand_features(df_filled, festival_dates=FESTIVAL_DATES)
print(f'Shape: {X.shape}  |  Mean y: {y.mean():.2f}')
print(f'NaNs: {X.isnull().sum().sum()}')

Filling missing dates...
Before: 844,338  After: 1,050,330
Building feature matrix...
Shape: (1016880, 15)  |  Mean y: 5605.50
NaNs: 0


## 5 — Train / Test Split (time-ordered)

In [25]:
feature_dates = df_filled.loc[X.index, 'date']
cutoff_date = feature_dates.quantile(0.80)

train_mask = feature_dates < cutoff_date
test_mask = ~train_mask

if train_mask.sum() == 0 or test_mask.sum() == 0:
    split_idx = int(len(X) * 0.80)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    print('Fallback split used: index-based 80/20')
else:
    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y.loc[train_mask], y.loc[test_mask]
    print(f'Date cutoff used: {cutoff_date.date()}')

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test,  label=y_test)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

Date cutoff used: 2015-01-24
Train: 812,949  Test: 203,931


## 6 — Cross-Validation

`early_stopping_rounds` is an argument to `xgb.cv()`, NOT a key in the params dict.

In [26]:
print('Running 5-fold CV...')
cv_result = xgb.cv(
    CV_PARAMS,
    dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    nfold=5,
    metrics='rmse',
    early_stopping_rounds=30,
    verbose_eval=100,
    seed=42,
)
best_n    = int(cv_result['test-rmse-mean'].idxmin()) + 1
best_rmse = cv_result['test-rmse-mean'].min()
print(f'Best num_boost_round: {best_n}  CV RMSE: {best_rmse:.4f}')

Running 5-fold CV...
[0]	train-rmse:3781.93480+1.01281	test-rmse:3782.04919+4.29940
[100]	train-rmse:1568.59789+1.31183	test-rmse:1579.31742+4.79723
[200]	train-rmse:1450.73171+2.48436	test-rmse:1469.93250+4.88969
[300]	train-rmse:1390.82397+4.25321	test-rmse:1416.92608+4.19495
[400]	train-rmse:1348.80888+2.67296	test-rmse:1381.45650+4.96929
[499]	train-rmse:1317.42522+2.20792	test-rmse:1355.95122+4.64077
Best num_boost_round: 500  CV RMSE: 1355.9512


## 7 — Final Train with Early Stopping

`early_stopping_rounds` is an argument to `xgb.train()`, NOT in params dict.

In [27]:
model = xgb.train(
    CV_PARAMS,
    dtrain,
    num_boost_round=best_n + 50,
    evals=[(dtrain, 'train'), (dtest, 'eval')],
    early_stopping_rounds=20,
    verbose_eval=50,
)
print(f'Best iteration: {model.best_iteration}  Best score: {model.best_score:.4f}')

[0]	train-rmse:3781.94673	eval-rmse:3799.44400
[50]	train-rmse:1735.16203	eval-rmse:1744.18404
[100]	train-rmse:1577.49179	eval-rmse:1590.00229
[150]	train-rmse:1497.60701	eval-rmse:1514.63615
[200]	train-rmse:1451.15548	eval-rmse:1471.28764
[250]	train-rmse:1419.16953	eval-rmse:1442.12859
[300]	train-rmse:1393.94970	eval-rmse:1419.23055
[350]	train-rmse:1372.57737	eval-rmse:1400.44048
[400]	train-rmse:1353.72107	eval-rmse:1383.95265
[450]	train-rmse:1334.61211	eval-rmse:1366.87041
[500]	train-rmse:1317.46634	eval-rmse:1352.11016
[549]	train-rmse:1307.28342	eval-rmse:1344.36908
Best iteration: 549  Best score: 1344.3691


## 8 — Evaluate

In [28]:
y_pred = np.clip(model.predict(dtest), 0, None)
mask   = y_test > 0
mape   = mean_absolute_percentage_error(y_test[mask], y_pred[mask])
print(f'Test MAPE : {mape:.4f} ({mape:.1%})')
print(f'Target    : <= {MAPE_TARGET:.0%}')
print(f'Result    : {"PASS" if mape <= MAPE_TARGET else "FAIL"}')

Test MAPE : 0.1202 (12.0%)
Target    : <= 15%
Result    : PASS


## 9 — Feature Importance

In [19]:
importance = pd.Series(model.get_score(importance_type='gain')).sort_values(ascending=False)
print('Feature importance (gain):')
print(importance.to_string())

Feature importance (gain):
day_of_week_sin         795374912.0
day_of_week_cos         137846704.0
rolling_avg_30          135392880.0
rolling_std_7           119255744.0
product_category_enc     99301384.0
rolling_avg_7            83550520.0
month_cos                83385848.0
depot_region_enc         78916776.0
month_sin                73286800.0
lag_14                   71391024.0
lag_30                   69424304.0
season_code              68245816.0
lag_7                    60461100.0


## 10 — Save to Registry

Only runs if MAPE target is met. Never call `joblib.dump()` directly.

In [30]:
store = ModelStore()
if mape <= MAPE_TARGET:
    try:
        saved_path = store.save(
            model,
            model_name='demand_global',
            metadata={
                'mape':           round(mape, 4),
                'best_iteration': model.best_iteration,
                'n_features':     len(FEATURE_COLS),
                'feature_cols':   FEATURE_COLS,
                'sample_frac':    SAMPLE_FRAC,
                'xgb_params':     CV_PARAMS,
                'dataset':        'rossmann',
            },
        )
        print(f'Model saved: {saved_path}')
        print(f'MAPE: {mape:.4f} — PASS (target <= {MAPE_TARGET:.2f})')
    except Exception as e:
        print(f'MAPE: {mape:.4f} — PASS (target <= {MAPE_TARGET:.2f})')
        print(f'Save skipped: {e}')
        print('Configure AWS_ACCESS_KEY/AWS_SECRET_KEY/S3_BUCKET to persist model.')
else:
    print(f'MAPE: {mape:.4f} — FAIL (target <= {MAPE_TARGET:.2f}); skipping save.')

MAPE: 0.1202 — PASS (target <= 0.15)
Save skipped: Unable to locate credentials
Configure AWS_ACCESS_KEY/AWS_SECRET_KEY/S3_BUCKET to persist model.
